# 02 Missing Data & Feature Engineering

Loads the raw CSVs saved in notebook 1, engineers event-level features 
from the raw columns, aggregates to driver-season level, identifies missing 
values, and applies imputation to produce a clean processed dataset.

**Input:** `data/raw/race_results_2021_2025.csv`, `data/raw/qualifying_results_2021_2025.csv`  
**Output:** `data/processed/driver_season_summary.csv`

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

In [4]:
RAW_DIR       = Path('../data/raw')
PROCESSED_DIR = Path('../data/processed')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

race_df  = pd.read_csv(RAW_DIR / 'race_results_2021_2025.csv')
quali_df = pd.read_csv(RAW_DIR / 'qualifying_results_2021_2025.csv')

print("Race data:  ", race_df.shape)
print("Quali data: ", quali_df.shape)
print("\nRace columns:", list(race_df.columns))

Race data:   (2278, 12)
Quali data:  (2277, 12)

Race columns: ['season', 'round', 'race_name', 'date', 'driver_id', 'driver_code', 'driver_name', 'constructor', 'grid', 'finish_position', 'points', 'status']


## Feature Engineering


In [5]:
race_df['is_win']    = (race_df['finish_position'] == 1).astype(int)
race_df['is_podium'] = race_df['finish_position'].isin([1, 2, 3]).astype(int)
race_df['is_points'] = (race_df['finish_position'] <= 10).astype(int)
race_df['is_dnf']    = (~race_df['status'].str.contains(
                            'Finished', case=False, na=False)).astype(int)
race_df['positions_gained'] = race_df['grid'] - race_df['finish_position']

print("Features added. Sample:")
race_df[['driver_name', 'season', 'round', 'grid',
         'finish_position', 'is_win', 'is_podium',
         'is_dnf', 'positions_gained']].head(8)

Features added. Sample:


,driver_name,season,round,grid,finish_position,is_win,is_podium,is_dnf,positions_gained
0,Lewis Hamilton,2021,1,2,1,1,1,0,1
1,Max Verstappen,2021,1,1,2,0,1,0,-1
2,Valtteri Bottas,2021,1,3,3,0,1,0,0
3,Lando Norris,2021,1,7,4,0,0,0,3
4,Sergio Pérez,2021,1,0,5,0,0,0,-5
5,Charles Leclerc,2021,1,4,6,0,0,0,-2
6,Daniel Ricciardo,2021,1,6,7,0,0,0,-1
7,Carlos Sainz,2021,1,8,8,0,0,0,0
